# Dynamic Bayesian Network â€” Machine-Failure Prediction from Alarm Data

> **All-alarms variant.** This copy watches **all 94 alarms** instead of the top 3.
> A pure DBN with `DBNInference` cannot do this for forward prediction (the next
> `State` would need ~190 parents), so the alarms are treated as **conditionally
> independent given the state** (Naive-Bayes) and the model is the equivalent flat
> network, still trained with **Maximum Likelihood Estimation (MLE)**.

**Objective:** Predict whether a machine will transition to a **Failure** state in
the next time window, given its current state and the alarm behaviour observed in
the current window.

$$P(\text{State}_{t+1} = \text{Failure} \mid \text{State}_t,\ \text{alarm features at } t)$$

### 1. What is a Bayesian Network (BN)?
A Bayesian Network is a probabilistic graphical model that represents a set of random variables and their conditional dependencies through a Directed Acyclic Graph (DAG). Grounded in Bayes' Theorem, it lets to compactly represent the joint probability distribution of an entire system by exploiting the conditional independencies between variables. Bayesian Networks are widely used for diagnostic and predictive reasoning under uncertainty.

### 2. What is a Dynamic Bayesian Network (DBN)?
A Dynamic Bayesian Network extends the traditional Bayesian Network to model temporal, sequential, or time-series data. While a standard BN captures a static "snapshot" of a system, a DBN connects multiple BNs sequentially across discrete time slices. It assumes the Markov property, the state of the system at time $t$ depends only on the state at time $t-1$, which keeps the model computationally tractable.

### 3. How are temporal dependencies represented in a DBN?
Temporal dependencies are modelled with **transition edges** (also called inter-slice edges) that cross from one time slice to the next. For example, a directed edge connects a node at time $t$ (e.g. `Machine_State_t`) to a node at time $t+1$ (`Machine_State_t+1`). These cross-slice arrows explicitly capture how past observations and history influence the future state.

### 4. What are Nodes, Directed Edges, and Conditional Probability Tables (CPTs)?
- **Nodes:** the building blocks of the graph, each representing a random variable. In this exercise the nodes represent variables such as *Alarm Count*, *Alarm Duration*, and the *Machine State* (Running / Failure).
- **Directed Edges:** arrows connecting nodes to describe influence or direct conditional dependency. An arrow from node $A$ to node $B$ means that $B$ is probabilistically conditioned on $A$.
- **Conditional Probability Tables (CPTs):** tables attached to each node that quantify the probability distribution of that node given every possible combination of its parents' states.

### Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from pgmpy.models import DynamicBayesianNetwork as DBN
try:
    from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
except ImportError:
    from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

print("Libraries imported successfully!")

Libraries imported successfully!


### Calulation of the alarm durations

In [2]:
df=pd.read_csv('dataset_exercise.csv', delimiter=';')

#Convert to datetime
df['start_alarm']=pd.to_datetime(df['start_alarm'])
df['end_alarm']=pd.to_datetime(df['end_alarm'])

#Duration of the alarms
df['duration_seconds']= (df['end_alarm']-df['start_alarm']).dt.total_seconds()
df.head()


,start_alarm,end_alarm,alarm_id,machine_state,time_window,duration_seconds
0,2025-06-23 03:08:36.666,2025-06-23 03:08:38.663,156700113,Failure,1,1.997
1,2025-06-23 03:08:36.666,2025-06-23 03:08:38.663,156700114,Failure,1,1.997
2,2025-06-23 03:08:42.154,2025-06-23 03:10:12.667,156701901,Failure,1,90.513
3,2025-06-23 03:08:42.669,2025-06-23 03:10:12.667,156702002,Failure,1,89.998
4,2025-06-23 03:08:42.669,2025-06-23 03:10:12.667,156701902,Failure,1,89.998


### Functions based on Step1 for alarm count and duration

In [3]:
def alarm_count_type(count):
    if count == 0: return 'None'
    elif 1 <= count <= 2: return 'Low'
    elif 3 <= count <= 5: return 'Medium'
    else: return 'High'

def alarm_duration_type(duration):
    if duration == 0: return 'None'
    elif 1 <= duration <= 30: return 'Short'
    elif 31 <= duration <= 300: return 'Medium'
    else: return 'Long'

### Get the alarms

In [4]:
top_alarms = df['alarm_id'].value_counts().index.tolist()   # ALL alarms
print(f"Using all {len(top_alarms)} alarm IDs as features")

Using all 94 alarm IDs as features


### Function for Step 2 builing a feature row per time windows
Compute each alarm's cound and duration, converting them to the categories in Step1 and label them with the major machine state in the window

In [5]:
def build_window_features(df, alarm_ids):
    processed_data = []
    for w, window_data in df.groupby('time_window'):
        state = window_data['machine_state'].mode().iloc[0]   # majority label
        row   = {'time_window': w, 'State': state}
        for i, alarm in enumerate(top_alarms, start=1):
            sub = window_data[window_data['alarm_id'] == alarm]
            row[f'A{i}_Count']    = alarm_count_type(len(sub))
            row[f'A{i}_Duration'] = alarm_duration_type(sub['duration_seconds'].sum())
        processed_data.append(row)
    return (pd.DataFrame(processed_data)
              .sort_values('time_window')
              .reset_index(drop=True))

### Apply the function

In [6]:
df_processed = build_window_features(df, top_alarms)

print(f"{len(df_processed)} windows built")
print("State distribution:", df_processed['State'].value_counts().to_dict())
df_processed.head()

329 windows built
State distribution: {'Running': 207, 'Failure': 122}


,time_window,State,A1_Count,A1_Duration,A2_Count,A2_Duration,A3_Count,A3_Duration,A4_Count,A4_Duration,...,A90_Count,A90_Duration,A91_Count,A91_Duration,A92_Count,A92_Duration,A93_Count,A93_Duration,A94_Count,A94_Duration
0,1,Failure,Medium,Long,Medium,Long,Medium,Long,Medium,Short,...,None,None,None,None,None,None,None,None,None,None
1,2,Failure,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,4,Failure,Medium,Long,Medium,Long,Medium,Long,Medium,Short,...,None,None,None,None,None,None,None,None,None,None
3,5,Running,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Short,...,None,None,None,None,None,None,None,None,None,None
4,6,Running,Medium,Medium,Medium,Medium,Medium,Medium,Low,Short,...,None,None,None,None,None,None,None,None,None,None


### Step3 Generate the training transition
The DBN learns from pairs of consecutive windows: the time t paired with t+1
I use `shift(-1)` to pair each window with the next, and filter out gaps because there are gaps in the `df['time_window']` column

In [7]:
df_processed['State_next']   = df_processed['State'].shift(-1)
df_processed['next_window']  = df_processed['time_window'].shift(-1)

#Check if next window is consecutive
df_transitions=df_processed[df_processed['next_window']==df_processed['time_window']+1].dropna().copy()

print(f"{len(df_transitions)} valid consecutive transitions")
print("Next-state distribution:", df_transitions['State_next'].value_counts().to_dict())
df_transitions.head()

195 valid consecutive transitions
Next-state distribution: {'Running': 119, 'Failure': 76}


,time_window,State,A1_Count,A1_Duration,A2_Count,A2_Duration,A3_Count,A3_Duration,A4_Count,A4_Duration,...,A91_Count,A91_Duration,A92_Count,A92_Duration,A93_Count,A93_Duration,A94_Count,A94_Duration,State_next,next_window
0,1,Failure,Medium,Long,Medium,Long,Medium,Long,Medium,Short,...,None,None,None,None,None,None,None,None,Failure,2.0
2,4,Failure,Medium,Long,Medium,Long,Medium,Long,Medium,Short,...,None,None,None,None,None,None,None,None,Running,5.0
3,5,Running,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Short,...,None,None,None,None,None,None,None,None,Running,6.0
6,20,Failure,Medium,Long,Medium,Long,Medium,Long,Low,Short,...,None,None,None,None,None,None,None,None,Failure,21.0
7,21,Failure,Medium,Long,Medium,Long,Medium,Long,Low,Short,...,None,None,None,None,None,None,None,None,Running,22.0


### Step 4 â€” Constructing the model

The conceptual temporal structure is declared with `DynamicBayesianNetwork` for
display (current alarms at *t* â†’ next state). For training, all 94 alarms only fit
under the **conditional-independence (Naive-Bayes)** assumption, so the computed
model is the equivalent flat network: `State_t â†’ State_t+1` and
`State_t+1 â†’ each alarm feature`.

In [8]:
alarm_cols = [c for c in df_transitions.columns
              if c.startswith('A') and (c.endswith('_Count') or c.endswith('_Duration'))]
Features = alarm_cols

dbn = DBN()
edges = [(('State', 0), ('State', 1))] + [((c, 0), ('State', 1)) for c in Features]
dbn.add_edges_from(edges)
print(f"DBN structure: State_t + {len(Features)} alarm features at t  ->  State at t+1")

DBN structure: State_t + 188 alarm features at t  ->  State at t+1


### Step 5 â€” Train the model (MLE)

`pgmpy`'s `DBN.fit` + `DBNInference` cannot represent ~190 parents for one `State`
node, so (as in the BDeu notebook) the model is the equivalent flat network with the
**conditional-independence (Naive-Bayes)** structure. It is trained with **Maximum
Likelihood Estimation** on all transitions, then queried with Variable Elimination.

In [9]:
Count_categs    = ['None', 'Low', 'Medium', 'High']
Duration_categs = ['None', 'Short', 'Medium', 'Long']
States        = ['Running', 'Failure']

model = BayesianNetwork([('State_t', 'State_t1')] +
                        [('State_t1', f'{c}_t') for c in Features])

data = df_transitions.rename(columns={
    'State': 'State_t', 'State_next': 'State_t1',
    **{c: f'{c}_t' for c in Features}
})

state_names = {'State_t': States, 'State_t1': States}
for c in Features:
    state_names[f'{c}_t'] = Count_categs if c.endswith('Count') else Duration_categs

cols = ['State_t', 'State_t1'] + [f'{c}_t' for c in Features]
mle  = MaximumLikelihoodEstimator(model, data[cols], state_names=state_names)
model.add_cpds(*mle.get_parameters())
print("Model trained with MLE. Valid:", model.check_model())

Model trained with MLE. Valid: True


### Expected Outcome/Inference
Use the example in the instructions to test the network

In [10]:
infer = VariableElimination(model)

def query(state_t, alarm_features):
    """Returns P(State_t+1 = Failure | evidence)."""
    ev = {'State_t': state_t}
    ev.update({f'{k}_t': v for k, v in alarm_features.items()})
    try:
        q = infer.query(['State_t1'], evidence=ev, show_progress=False)
        return float(q.values[q.state_names['State_t1'].index('Failure')])
    except Exception:
        return 0.5


example = {'A1_Count': 'High', 'A1_Duration': 'Long',
           'A2_Count': 'Medium', 'A2_Duration': 'Medium'}
print("P(Failure | State=Running, example alarms) =", round(query('Running', example), 3))

P(Failure | State=Running, example alarms) = 0.043


### Model evaluation and metrics

In [11]:
tp = fp = fn = tn = 0
for _, r in data.iterrows():
    feats = {c: r[f'{c}_t'] for c in Features}
    pred  = 'Failure' if query(r['State_t'], feats) >= 0.5 else 'Running'
    true  = r['State_t1']
    tp += (true == 'Failure' and pred == 'Failure')
    fp += (true == 'Running' and pred == 'Failure')
    fn += (true == 'Failure' and pred == 'Running')
    tn += (true == 'Running' and pred == 'Running')

total = tp + fp + fn + tn
prec = tp / (tp + fp) if (tp + fp) else 0.0
rec  = tp / (tp + fn) if (tp + fn) else 0.0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
print(f"Total Evaluated: {total}")
print(f"TP: {tp} | FP: {fp}")
print(f"TN: {tn} | FN: {fn}")
print(f"Accuracy : {(tp + tn) / total:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall   : {rec:.3f}")
print(f"F1 Score : {f1:.3f}")

Total Evaluated: 195
TP: 47 | FP: 3
TN: 116 | FN: 29
Accuracy : 0.836
Precision: 0.940
Recall   : 0.618
F1 Score : 0.746
